# Healthcare Sector Statistical Analysis Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

df_raw = pd.read_csv("healthcare_raw_dataset.csv")
display(df_raw.head())
display(df_raw.tail())
print("Rows and columns:", df_raw.shape)
print("Data types:\n", df_raw.dtypes)
print("Missing values:\n", df_raw.isna().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
categorical = df_raw.select_dtypes(include=["object"]).columns.tolist()
numerical = df_raw.select_dtypes(include=np.number).columns.tolist()
print("Categorical:", categorical)
print("Numerical:", numerical)

df = df_raw.drop_duplicates().copy()
numeric_cols = ["age", "bmi", "blood_pressure", "heart_rate", "hospital_days", "treatment_cost"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.loc[~df["age"].between(18, 100), "age"] = np.nan
df.loc[~df["bmi"].between(10, 70), "bmi"] = np.nan
df.loc[~df["blood_pressure"].between(60, 250), "blood_pressure"] = np.nan
df.loc[~df["heart_rate"].between(30, 220), "heart_rate"] = np.nan
df.loc[~df["hospital_days"].between(1, 365), "hospital_days"] = np.nan
df.loc[df["treatment_cost"] <= 0, "treatment_cost"] = np.nan
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in ["gender", "smoker", "diabetes", "follow_up"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df["age"] = df["age"].round().astype(int)
df["hospital_days"] = df["hospital_days"].round().astype(int)
print("Cleaned shape:", df.shape)
print("Remaining missing values:\n", df.isna().sum())
print("Remaining duplicates:", df.duplicated().sum())
df.to_csv("healthcare_cleaned_dataset.csv", index=False)
display(df.head())

In [ ]:
selected = ["age", "bmi", "hospital_days", "treatment_cost", "blood_pressure", "heart_rate"]
summary = pd.DataFrame(index=selected)
for col in selected:
    s = df[col]
    summary.loc[col, "mean"] = s.mean()
    summary.loc[col, "median"] = s.median()
    summary.loc[col, "mode"] = s.mode()[0]
    summary.loc[col, "min"] = s.min()
    summary.loc[col, "max"] = s.max()
    summary.loc[col, "range"] = s.max() - s.min()
    summary.loc[col, "variance"] = s.var()
    summary.loc[col, "std_dev"] = s.std()
    summary.loc[col, "Q1"] = s.quantile(.25)
    summary.loc[col, "Q3"] = s.quantile(.75)
    summary.loc[col, "IQR"] = s.quantile(.75) - s.quantile(.25)
display(summary.round(3))
display(df[selected].describe())

In [ ]:
for col in ["age", "treatment_cost"]:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col], bins=18, edgecolor="black")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col); plt.ylabel("Number of patients")
    plt.show()
for col in ["hospital_days", "treatment_cost"]:
    plt.figure(figsize=(7, 4))
    plt.boxplot(df[col])
    plt.title(f"Box plot of {col}")
    plt.ylabel(col)
    plt.show()
for col in ["age", "bmi", "hospital_days", "treatment_cost"]:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: IQR fences=({lower:.2f}, {upper:.2f}); possible outliers={count}")

In [ ]:
events = {
    "Smoker": (df["smoker"] == "Yes").mean(),
    "Hospital stay > 5 days": (df["hospital_days"] > 5).mean(),
    "Diabetes": (df["diabetes"] == "Yes").mean(),
    "Follow-up required": (df["follow_up"] == "Yes").mean(),
    "BMI > 30": (df["bmi"] > 30).mean()
}
for event, probability in events.items():
    print(f"P({event}) = {probability:.3f} = {probability:.1%}")

In [ ]:
yes = df.loc[df["follow_up"] == "Yes", "hospital_days"]
no = df.loc[df["follow_up"] == "No", "hospital_days"]
test = stats.ttest_ind(yes, no, equal_var=False)
print("Follow-up yes: n =", len(yes), "mean =", yes.mean())
print("Follow-up no: n =", len(no), "mean =", no.mean())
print("t statistic =", test.statistic)
print("p-value =", test.pvalue)
print("Decision:", "Reject H0" if test.pvalue < .05 else "Fail to reject H0")

In [ ]:
values = df["treatment_cost"]
sample_mean = values.mean()
standard_error = stats.sem(values)
ci_low, ci_high = stats.t.interval(.95, len(values)-1, loc=sample_mean, scale=standard_error)
print("Sample mean:", sample_mean)
print("95% confidence interval:", (ci_low, ci_high))

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
display(corr.round(3))
plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()
plt.figure(figsize=(7, 4))
plt.scatter(df["age"], df["blood_pressure"], alpha=.6)
plt.title("Age vs Blood Pressure")
plt.xlabel("Age (years)"); plt.ylabel("Blood pressure (mmHg)")
plt.show()
print("Correlation (age, blood pressure):", corr.loc["age", "blood_pressure"])
print("Correlation does not prove causation.")

In [ ]:
X = df[["hospital_days"]]
y = df["treatment_cost"]
model = LinearRegression().fit(X, y)
predictions = model.predict(X)
r2 = r2_score(y, predictions)
rmse = mean_squared_error(y, predictions) ** 0.5
print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])
print(f"Equation: treatment_cost = {model.intercept_:.2f} + {model.coef_[0]:.2f} * hospital_days")
print("Predicted cost for 5 days:", model.predict(pd.DataFrame({"hospital_days": [5]}))[0])
print("R-squared:", r2)
print("RMSE:", rmse)
plt.figure(figsize=(7, 4))
plt.scatter(df["hospital_days"], y, alpha=.65, label="Observed")
line_x = np.linspace(df["hospital_days"].min(), df["hospital_days"].max(), 100).reshape(-1, 1)
plt.plot(line_x, model.predict(line_x), label="Regression line")
plt.title("Hospital Days vs Treatment Cost")
plt.xlabel("Hospital days"); plt.ylabel("Treatment cost")
plt.legend(); plt.show()

## Final questions — guide to your answers
1. The dataset describes fictional patients using demographic, health, admission, cost and follow-up variables.
2. Compare the variance column in the descriptive-statistics table. Remember that variables have different units.
3. Use the IQR outlier counts printed in Task 3.
4. Compare the probabilities printed in Task 4.
5. H0 states equal mean hospital stays in the two follow-up groups.
6. Use the p-value from Task 5; it measures evidence against H0 under the test assumptions.
7. The confidence interval gives a range of plausible values for the population mean under the method.
8. Find the largest absolute off-diagonal value in the correlation matrix.
9. No. Correlation alone does not demonstrate causation.
10. The regression predicts treatment cost from hospital days.
11. R² reports the proportion of sample variation explained by the one-predictor linear model.
12. Limitations include simulated data, sample size, imputation, omitted variables and model assumptions.

**Executive summary:** The report document includes a 150–250-word executive summary based on the computed results.

**Reproducibility:** The dataset is fictional and contains no real patient information.

## Answers to the 12 required questions

1. What does the dataset tell you about the patients? It describes 320 fictional clinic records across age, gender, BMI, blood pressure, heart rate, hospital stay, treatment cost, smoking, diabetes and follow-up status. The average age is 51.85 years.

2. Which numerical variable has the greatest variation? Treatment cost has the largest numerical variance (1,305,928.83 fictional-units squared). Variances across different units should be compared cautiously.

3. Which variables contain possible outliers? The IQR method flagged age (9), BMI (2), hospital_days (5) and treatment_cost (4) values.

4. Which probability result did you find most interesting, and why? Follow-up was required for 66.2% of records, the largest of the listed event probabilities. This is interesting because follow-up is common in this simulated sample.

5. What was your null hypothesis? H0: The population mean hospital stay is equal for patients who require follow-up and those who do not.

6. What was your p-value, and what does it mean? The Welch t-test p-value was 8.70531e-11. Under H0 and the test assumptions, this is the probability of observing a difference at least as extreme as the one measured. Since it is below 0.05, the test rejects H0.

7. What does your 95% confidence interval tell you? The interval for mean treatment cost is 3,889.01 to 4,140.38 fictional units. It describes uncertainty around the estimated mean under the t-interval method.

8. Which variables have the strongest correlation? Hospital_days and treatment_cost have the strongest absolute correlation, r = 0.858, a strong positive linear association.

9. Does the correlation prove that one variable causes the other? No. Correlation alone cannot establish causation; other variables or the simulation design may explain the association.

10. What does your regression model predict? It predicts treatment cost from hospital stay: cost = 776.37 + 642.44 × hospital_days. For 5 hospital days, predicted cost is 3,988.59 fictional units.

11. How well does the regression model explain the target variable? R² = 0.736, so hospital_days explains about 73.6% of the variation in treatment_cost in this sample.

12. What are the biggest limitations? The data are simulated, the sample is limited, missing values were imputed, important variables such as diagnosis and illness severity are absent, and correlation/regression do not establish medical causes.